# Sprite Converter — Pillow

Outils de conversion entre GIF, spritesheets et images PNG individuelles.

**Convention :** les coordonnées `(x, y)` sont les positions du coin supérieur gauche de chaque sprite. Les dimensions des sprites sont fournies séparément lors du découpage.

Les images sont manipulées en RGBA afin de préserver la transparence.

In [ ]:
from pathlib import Path
from PIL import Image

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

def _save_png(image, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    image.convert('RGBA').save(path, format='PNG')
    return str(path)

def _load_pngs(images):
    loaded = []
    for item in images:
        if isinstance(item, Image.Image):
            loaded.append(item.convert('RGBA').copy())
        else:
            with Image.open(item) as im:
                loaded.append(im.convert('RGBA').copy())
    if not loaded:
        raise ValueError('La liste d’images est vide.')
    return loaded

def _check_same_size(images):
    sizes = {im.size for im in images}
    if len(sizes) != 1:
        raise ValueError(f'Toutes les images doivent avoir les mêmes dimensions. Tailles trouvées : {sizes}')
    return images[0].size

def grid_positions(n_row, n_column, sprite_width, sprite_height):
    """Retourne les coordonnées (x, y) des cellules, de gauche à droite puis de haut en bas."""
    if min(n_row, n_column, sprite_width, sprite_height) <= 0:
        raise ValueError('Les dimensions doivent être strictement positives.')
    return [(col * sprite_width, row * sprite_height)
            for row in range(n_row) for col in range(n_column)]

## 1. GIF → Spritesheet

In [ ]:
def gif_to_spritesheet(n_row, n_column, gif_path, output_path=None, spacing=0, background=(0, 0, 0, 0)):
    """Extrait les frames d'un GIF et les place dans une grille.

    Les cases sont remplies dans l'ordre de lecture : gauche à droite, puis ligne suivante.
    Si la grille contient trop de cases, les dernières restent transparentes.
    """
    if n_row <= 0 or n_column <= 0 or spacing < 0:
        raise ValueError('Dimensions de grille invalides ou espacement négatif.')
    with Image.open(gif_path) as gif:
        frames = []
        for i in range(getattr(gif, 'n_frames', 1)):
            gif.seek(i)
            frames.append(gif.convert('RGBA').copy())
    if len(frames) > n_row * n_column:
        raise ValueError(f'La grille contient {n_row*n_column} cases, mais le GIF a {len(frames)} frames.')
    frame_w, frame_h = _check_same_size(frames)
    sheet = Image.new('RGBA',
        (n_column * frame_w + (n_column - 1) * spacing,
         n_row * frame_h + (n_row - 1) * spacing), background)
    for i, frame in enumerate(frames):
        x = (i % n_column) * (frame_w + spacing)
        y = (i // n_column) * (frame_h + spacing)
        sheet.alpha_composite(frame, (x, y))
    if output_path is None:
        output_path = OUTPUT_DIR / f'{Path(gif_path).stem}_spritesheet.png'
    return _save_png(sheet, output_path)

## 2. Spritesheet → GIF

In [ ]:
def spritesheet_to_gif(coordinates, spritesheet_path, sprite_size, output_path=None, duration=100, loop=0, spacing=0):
    """Découpe les sprites aux coordonnées indiquées et les assemble en GIF.

    sprite_size = (largeur, hauteur). duration est exprimée en millisecondes.
    """
    sprite_w, sprite_h = sprite_size
    if sprite_w <= 0 or sprite_h <= 0 or duration <= 0 or spacing < 0:
        raise ValueError('Dimensions, durée ou espacement invalides.')
    frames = []
    with Image.open(spritesheet_path) as sheet:
        sheet = sheet.convert('RGBA')
        for x, y in coordinates:
            if x < 0 or y < 0 or x + sprite_w > sheet.width or y + sprite_h > sheet.height:
                raise ValueError(f'Sprite hors de la spritesheet : {(x, y)}')
            frames.append(sheet.crop((x, y, x + sprite_w, y + sprite_h)))
    if not frames:
        raise ValueError('Aucune coordonnée fournie.')
    if output_path is None:
        output_path = OUTPUT_DIR / f'{Path(spritesheet_path).stem}.gif'
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    # GIF ne gère pas l'alpha continu : Pillow convertit la transparence en palette GIF.
    frames[0].save(output_path, save_all=True, append_images=frames[1:],
                   duration=duration, loop=loop, disposal=2, transparency=0)
    return str(output_path)

## 3. GIF → PNG individuels

In [ ]:
def gif_to_png(gif_path, output_dir=None, prefix=None):
    """Extrait chaque frame d'un GIF dans un PNG. Retourne la liste des chemins créés."""
    gif_path = Path(gif_path)
    output_dir = Path(output_dir) if output_dir else OUTPUT_DIR / f'{gif_path.stem}_frames'
    output_dir.mkdir(parents=True, exist_ok=True)
    prefix = prefix or gif_path.stem
    output_paths = []
    with Image.open(gif_path) as gif:
        for i in range(getattr(gif, 'n_frames', 1)):
            gif.seek(i)
            frame = gif.convert('RGBA').copy()
            output_paths.append(_save_png(frame, output_dir / f'{prefix}_{i:03d}.png'))
    return output_paths

## 4. PNG individuels → GIF

In [ ]:
def png_to_gif(png_images, output_path='output/animation.gif', duration=100, loop=0):
    """Assemble une liste ordonnée de PNG (chemins ou objets PIL) en GIF animé."""
    if duration <= 0:
        raise ValueError('La durée doit être strictement positive.')
    frames = _load_pngs(png_images)
    _check_same_size(frames)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames[0].save(output_path, save_all=True, append_images=frames[1:],
                   duration=duration, loop=loop, disposal=2, transparency=0)
    return str(output_path)

## 5. PNG individuels → Spritesheet

In [ ]:
def png_to_spritesheet(n_row, n_column, png_images, output_path='output/spritesheet.png', spacing=0, background=(0, 0, 0, 0)):
    """Assemble une liste ordonnée de PNG dans une grille. Les cases inutilisées restent vides."""
    if n_row <= 0 or n_column <= 0 or spacing < 0:
        raise ValueError('Dimensions de grille invalides ou espacement négatif.')
    images = _load_pngs(png_images)
    image_w, image_h = _check_same_size(images)
    if len(images) > n_row * n_column:
        raise ValueError(f'La grille contient {n_row*n_column} cases, mais {len(images)} images ont été fournies.')
    sheet = Image.new('RGBA',
        (n_column * image_w + (n_column - 1) * spacing,
         n_row * image_h + (n_row - 1) * spacing), background)
    for i, image in enumerate(images):
        x = (i % n_column) * (image_w + spacing)
        y = (i // n_column) * (image_h + spacing)
        sheet.alpha_composite(image, (x, y))
    return _save_png(sheet, output_path)

## 6. Spritesheet → PNG individuels

In [ ]:
def spritesheet_to_png(coordinates, spritesheet_path, sprite_size, output_dir=None, prefix=None):
    """Extrait un PNG par coordonnée. Retourne la liste des chemins créés."""
    sprite_w, sprite_h = sprite_size
    if sprite_w <= 0 or sprite_h <= 0:
        raise ValueError('Dimensions de sprite invalides.')
    output_dir = Path(output_dir) if output_dir else OUTPUT_DIR / f'{Path(spritesheet_path).stem}_sprites'
    output_dir.mkdir(parents=True, exist_ok=True)
    prefix = prefix or Path(spritesheet_path).stem
    paths = []
    with Image.open(spritesheet_path) as sheet:
        sheet = sheet.convert('RGBA')
        for i, (x, y) in enumerate(coordinates):
            if x < 0 or y < 0 or x + sprite_w > sheet.width or y + sprite_h > sheet.height:
                raise ValueError(f'Sprite hors de la spritesheet : {(x, y)}')
            sprite = sheet.crop((x, y, x + sprite_w, y + sprite_h))
            paths.append(_save_png(sprite, output_dir / f'{prefix}_{i:03d}.png'))
    return paths

## 7. Recoloration

In [ ]:
# ---------------------------------------------------------------------------
# Palettes
# ---------------------------------------------------------------------------

SKIN = {
    "tres_claire": ("#F9D8C1", "#EFB89B", "#D98E73", "#FFE9DA"),
    "claire":      ("#F2C3A5", "#DFA080", "#C97960", "#FFE0CA"),
    "medium":      ("#C98768", "#A9654E", "#854636", "#E4A987"),
    "olive":       ("#B98161", "#956047", "#6E3F31", "#D59A75"),
    "brune":       ("#8D5238", "#6E392A", "#4D271F", "#B76F4E"),
    "tres_brune":  ("#633725", "#4A281D", "#301A16", "#854C34"),
}

HAIR = {
    "blond":       ("#F2C05C", "#DFA43B", "#B97922"),
    "blond_clair": ("#FFE39A", "#EBCB73", "#C5A34C"),
    "roux":        ("#F06B20", "#C84A16", "#8D2F12"),
    "chatain":     ("#74402D", "#55291F", "#381A17"),
    "brun":        ("#4A2920", "#321B18", "#211311"),
    "noir":        ("#2A211F", "#1D1716", "#111010"),
    "gris":        ("#D5C9A9", "#A99D7E", "#756C57"),
    "blanc":       ("#F0EDE0", "#CFC9B7", "#A8A18E"),
}

In [ ]:
baseFile=r"humans/skin/Female Skin1.png"
resultFile=r"humans/skin/female/female_spritesheet_xlight.png"

colorswap={"#fef0e4":SKIN["tres_claire"][3],"#f3d0C4":SKIN["tres_claire"][0],"#dea99f":SKIN["tres_claire"][1],"#b86a6a":SKIN["tres_claire"][2],"#e65656":SKIN["tres_claire"][2]}

def hex_to_rgb(value):
    value = value.lstrip('#')
    lv = len(value)
    return tuple(int(value[i:i + lv // 3], 16) for i in range(0, lv, lv // 3))

img = Image.open(baseFile)
pixels=img.load()

pixels = img.load() # create the pixel map

for i in range(img.size[0]): # for every pixel:
    for j in range(img.size[1]):
        if '#%02x%02x%02x' % pixels[i,j] in colorswap.keys:
            key='#%02x%02x%02x' % pixels[i,j]
            pixels[i,j] = hex_to_rgb(colorswap[key])

img.save(resultFile)


## Exemple d'utilisation

Pour une spritesheet composée de sprites de 64 × 64 pixels, avec 4 colonnes et 3 lignes :

In [ ]:
coords = grid_positions(n_row=3, n_column=4, sprite_width=64, sprite_height=64)

# GIF -> spritesheet
# sheet_path = gif_to_spritesheet(3, 4, 'animation.gif')

# Spritesheet -> PNG individuels
# png_paths = spritesheet_to_png(coords, 'spritesheet.png', (64, 64))

# Spritesheet -> GIF
# gif_path = spritesheet_to_gif(coords, 'spritesheet.png', (64, 64), duration=120)

# PNG individuels -> spritesheet
# sheet_path = png_to_spritesheet(3, 4, png_paths)

# PNG individuels -> GIF
# gif_path = png_to_gif(png_paths, duration=120)

# GIF -> PNG individuels
# png_paths = gif_to_png('animation.gif')